# 🤗 Hugging Face Playground

A lightweight notebook for running inference with Hugging Face Transformers models locally on Mac (CPU).

**Models used:** GPT-2, DistilBERT, sentence-transformers  
**Setup:** Make sure you're using the `.venv` kernel from this directory.

## 1. Environment Check
Verify that all packages are installed and check the compute device.

In [ ]:
import torch
import transformers
import numpy as np

print(f"PyTorch version:      {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"NumPy version:        {np.__version__}")
print(f"Device:               {'MPS (Apple GPU)' if torch.backends.mps.is_available() else 'CPU'}")
print(f"MPS available:        {torch.backends.mps.is_available()}")

## 2. Text Generation with GPT-2
Generate text using the GPT-2 model via the `pipeline` API.

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

prompt = "The future of artificial intelligence is"
results = generator(prompt, max_length=100, num_return_sequences=2, do_sample=True, temperature=0.8)

for i, result in enumerate(results):
    print(f"--- Generation {i+1} ---")
    print(result["generated_text"])
    print()

## 3. Sentiment Analysis
Classify text sentiment using a pre-trained DistilBERT model.

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

texts = [
    "I absolutely love this new framework, it's amazing!",
    "This is terrible and completely broken.",
    "The weather is okay today, nothing special.",
    "Scaling laws show that larger models perform predictably better."
]

results = classifier(texts)
for text, result in zip(texts, results):
    print(f"{result['label']} ({result['score']:.3f}): {text}")

## 4. Token-Level Exploration
Understand how tokenization works under the hood.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Scaling laws for neural language models show power-law relationships."
tokens = tokenizer.encode(text)
decoded_tokens = [tokenizer.decode([t]) for t in tokens]

print(f"Original text: {text}")
print(f"Number of tokens: {len(tokens)}")
print(f"Token IDs: {tokens}")
print(f"Decoded tokens: {decoded_tokens}")

## 5. Text Embeddings
Extract dense vector representations from text using a small model.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

sentences = [
    "The cat sat on the mat.",
    "A kitten rested on the rug.",
    "Stock prices rose sharply today."
]

def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze()

embeddings = [get_embedding(s) for s in sentences]

# Compute cosine similarity between pairs
from torch.nn.functional import cosine_similarity
print(f"Similarity (cat vs kitten):  {cosine_similarity(embeddings[0], embeddings[1], dim=0):.4f}")
print(f"Similarity (cat vs stocks):  {cosine_similarity(embeddings[0], embeddings[2], dim=0):.4f}")
print(f"Similarity (kitten vs stocks): {cosine_similarity(embeddings[1], embeddings[2], dim=0):.4f}")

## 6. Fill-Mask (Masked Language Modeling)
Predict masked tokens in a sentence using BERT.

In [ ]:
from transformers import pipeline

fill_mask = pipeline("fill-mask", model="distilbert-base-uncased")

results = fill_mask("Artificial intelligence will [MASK] the world.")
for r in results:
    print(f"{r['token_str']:>15s}  (score: {r['score']:.4f})  ->  {r['sequence']}")

## 7. Question Answering
Extract answers from a given context passage.

In [ ]:
from transformers import pipeline

qa = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

context = """
Scaling laws for neural language models, published by Kaplan et al. in 2020,
demonstrated that language model performance follows smooth power-law relationships
with model size, dataset size, and compute budget. The paper found that model size
matters most for a fixed compute budget, and that architectural details like width
and depth have minimal impact as long as total parameter count is held constant.
"""

questions = [
    "Who published the scaling laws paper?",
    "What relationships did the paper find?",
    "What matters most for a fixed compute budget?"
]

for q in questions:
    result = qa(question=q, context=context)
    print(f"Q: {q}")
    print(f"A: {result['answer']}  (confidence: {result['score']:.3f})")
    print()